# PAS Sample Prediction and Clustering

This notebook reruns the sample workflow:

1. Extract PAS feature vectors for the 1000-row sample, train split, and test split.
2. Train prediction models on the 800-row train split and evaluate on the 200-row test split.
3. Run clustering on the 1000-row sample feature vectors.
4. Generate result visualizations.

The implementation lives in `scripts/` so the notebook is mostly an executable control panel.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "sample"

print(PROJECT_ROOT)

## Dependency Check

If anything is missing, install it once in your environment, then rerun the notebook. The core workflow currently uses `pandas`, `numpy`, `scikit-learn`, `matplotlib`, and optionally `xgboost`.

In [ ]:
packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "xgboost": "xgboost",
}

missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
print("Missing packages:", missing if missing else "none")

# Uncomment if you need to install missing packages from inside the notebook:
# if missing:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

## Current Input Files

These should already exist from the sample creation step.

In [ ]:
sample_files = [
    DATA_DIR / "compas-2x_pastries_features_sample.csv",
    DATA_DIR / "compas-2x_pastries_features_sample_train.csv",
    DATA_DIR / "compas-2x_pastries_features_sample_test.csv",
]

for path in sample_files:
    frame = pd.read_csv(path)
    print(f"{path.name}: {len(frame)} rows, {len(frame.columns)} columns")

## 1. Extract Feature Vectors

In [ ]:
subprocess.check_call([sys.executable, "scripts/extract_pas_features.py"], cwd=PROJECT_ROOT)

In [ ]:
feature_files = [
    DATA_DIR / "compas-2x_pastries_features_sample_feature_vectors.csv",
    DATA_DIR / "compas-2x_pastries_features_sample_train_feature_vectors.csv",
    DATA_DIR / "compas-2x_pastries_features_sample_test_feature_vectors.csv",
]

for path in feature_files:
    frame = pd.read_csv(path)
    print(f"{path.name}: {len(frame)} rows, {len(frame.columns)} columns")

pd.read_csv(feature_files[0]).head()

## 2. Train Prediction Models and Run Clustering

In [ ]:
subprocess.check_call([sys.executable, "scripts/run_sample_models.py"], cwd=PROJECT_ROOT)

## 3. Generate Visualizations

In [ ]:
subprocess.check_call([sys.executable, "scripts/visualize_sample_results.py"], cwd=PROJECT_ROOT)

## Prediction Metrics

In [ ]:
metrics = pd.read_csv(DATA_DIR / "sample_prediction_metrics_with_error_percent.csv")
metrics.sort_values(["target", "rmse"]).head(50)

In [ ]:
best_by_target = metrics.sort_values(["target", "rmse"]).groupby("target").first().reset_index()
best_by_target[["target", "model", "mae", "rmse", "r2", "mae_percent"]].sort_values("target")

## Top Feature Importances

These are available for tree models: Random Forest, Gradient Boosting, and XGBoost.

In [ ]:
top_features = pd.read_csv(DATA_DIR / "sample_top10_feature_importances.csv")
top_features.head(30)

In [ ]:
target = "gap"
top_features[top_features["target"] == target].sort_values(["model", "importance"], ascending=[True, False])

## Clustering Outputs

In [ ]:
clusters = pd.read_csv(DATA_DIR / "sample_clusters.csv")
cluster_summary = pd.read_csv(DATA_DIR / "sample_cluster_summary.csv")
display(cluster_summary)
clusters.head()

## Visual Results

In [ ]:
for image_name in [
    "prediction_error_percent.png",
    "prediction_r2.png",
    "predicted_vs_actual_best_models.png",
    "cluster_pca_maps.png",
    "cluster_target_profiles.png",
]:
    print(image_name)
    display(Image(filename=str(RESULTS_DIR / image_name)))

In [ ]:
for target in ["gap", "homo", "lumo", "dipole_norm", "aip", "aea", "nfod"]:
    image_name = f"top_features_{target}.png"
    print(image_name)
    display(Image(filename=str(RESULTS_DIR / image_name)))